# Notebook 5 — Feature Engineering

## Objective

This notebook prepares the features required for machine learning models that predict whether an order will be delivered late or on time.

The feature engineering process is based on the findings from the EDA stage. The main objectives are to:

* Create predictive features using information available at prediction time.
* Avoid data leakage from future information.
* Handle missing values appropriately.
* Encode categorical variables.
* Scale numerical features when required.
* Fit preprocessing transformations on the training data only.
* Apply the fitted transformations consistently to the validation and test datasets.
* Save the fitted preprocessing objects, final feature table, and feature list for reuse in model training and inference.

### Input Data

The notebook uses the train, validation, and test datasets created in Notebook 3.

### Output Artifacts

The following artifacts will be generated:

* Final preprocessed feature table containing the training, validation, and test observations.
* Fitted preprocessing transformers, including imputers, encoders, and scalers.
* Feature list documenting the final model input features.

## Load Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("../Artifacts/train.csv")
validation = pd.read_csv("../Artifacts/validation.csv")
test = pd.read_csv("../Artifacts/test.csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67533, 14)
Validation shape: (14471, 14)
Test shape: (14472, 14)


In [3]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'number_of_items',
 'total_price',
 'total_freight',
 'number_of_payments',
 'total_payment_value',
 'delivery_label']

## Feature Engineering

The goal of this notebook is to transform the train, validation, and test datasets into model-ready features while avoiding data leakage.

The features are based on the patterns identified during the EDA stage. Only information that would be available at prediction time is used.

### Features to be created

* **Number of items:** total number of items in the order.
* **Total price:** total price of the order items.
* **Total freight:** total freight value for the order.
* **Number of payments:** number of payment records associated with the order.
* **Total payment value:** total payment value associated with the order.
* **Purchase hour:** hour of the day when the order was placed.
* **Purchase weekday:** day of the week when the order was placed.
* **Purchase month:** month in which the order was placed.
* **Estimated delivery days:** number of days between the purchase timestamp and the estimated delivery date.
* **Distance:** estimated distance between the customer and seller locations.

### Leakage Prevention

Features that depend on events occurring after the prediction point, such as actual delivery duration, actual delivery dates, or other post-purchase outcomes, will not be used as model features.

All fitted preprocessing transformations will be learned using the training split only and then applied consistently to the validation and test splits.

In [4]:
for df in [train, validation, test]:
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )
    
    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

In [5]:
train[[
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]].dtypes

order_purchase_timestamp         datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [6]:
for df in [train, validation, test]:
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
    
    df["purchase_weekday"] = (
        df["order_purchase_timestamp"].dt.day_name()
    )
    
    df["purchase_month"] = (
        df["order_purchase_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

In [7]:
train[
    [
        "purchase_hour",
        "purchase_weekday",
        "purchase_month"
    ]
].head()

,purchase_hour,purchase_weekday,purchase_month
0,12,Thursday,2016-09
1,9,Monday,2016-10
2,16,Monday,2016-10
3,21,Monday,2016-10
4,21,Monday,2016-10


In [8]:
for df in [train, validation, test]:
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

In [9]:
train["estimated_delivery_days"].describe()

count    67533.000000
mean        24.597288
std          8.022133
min          7.005127
25%         19.543333
50%         23.621377
75%         28.565984
max        155.135463
Name: estimated_delivery_days, dtype: float64

### Estimated Delivery Time

The `estimated_delivery_days` feature represents the expected number of days between the order purchase timestamp and the estimated delivery date.

This feature is available at prediction time because the estimated delivery date is associated with the order when it is placed. Therefore, it can be used as a predictive feature without relying on the actual delivery outcome.

The distribution of the feature will be reviewed for missing values and potential outliers before preprocessing.

In [10]:
train["estimated_delivery_days"].isna().sum()

np.int64(0)

No missing values were found in `estimated_delivery_days` in the training data, so no imputation is required for this feature.


In [11]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'number_of_items',
 'total_price',
 'total_freight',
 'number_of_payments',
 'total_payment_value',
 'delivery_label',
 'purchase_hour',
 'purchase_weekday',
 'purchase_month',
 'estimated_delivery_days']

### Geographic Features

The EDA identified customer geography and customer-seller distance as potentially useful predictive features.

These features are derived only from information available at prediction time:

* `customer_state` is obtained from the customer's location.
* `distance_km` is calculated from the customer and seller geographic coordinates.

The geographic source tables are loaded separately because these features were not included in the train, validation, and test artifacts created in the previous notebook.


In [12]:
customers = pd.read_csv("../Data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../Data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../Data/raw/olist_geolocation_dataset.csv")

print("Customers:", customers.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)

Customers: (99441, 5)
Sellers: (3095, 4)
Geolocation: (1000163, 5)


### Customer State

The `customer_state` feature represents the state where the customer is located.

Customer location is available at prediction time, so this feature can be used without introducing future information or data leakage.


In [13]:
train = train.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

validation = validation.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

test = test.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

In [14]:
train["customer_state"].isna().sum()

np.int64(0)

The `customer_state` feature was successfully added to the train, validation, and test datasets. No missing customer states were found in the training data.


### Geographic Coordinates by ZIP Prefix

The geolocation dataset contains multiple coordinate observations for the same ZIP code prefix. To obtain a single representative location for each prefix, the mean latitude and longitude are calculated.
This aggregated lookup table will be used to derive customer and seller coordinates and calculate the geographical distance between them.


In [15]:
geo_by_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)
geo_by_zip.head()

,geolocation_zip_code_prefix,latitude,longitude
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


### Customer Coordinates

Customer geographic coordinates are obtained by mapping each customer's ZIP code prefix to the corresponding aggregated geographic coordinates.

These coordinates are based on customer location information available at prediction time.


In [16]:
train = train.merge(
    customers[[
        "customer_id",
        "customer_zip_code_prefix"
    ]],
    on="customer_id",
    how="left"
)

validation = validation.merge(
    customers[[
        "customer_id",
        "customer_zip_code_prefix"
    ]],
    on="customer_id",
    how="left"
)

test = test.merge(
    customers[[
        "customer_id",
        "customer_zip_code_prefix"
    ]],
    on="customer_id",
    how="left"
)

In [17]:
train = train.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

validation = validation.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

test = test.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

In [18]:
train[["customer_zip_code_prefix", "latitude", "longitude"]].isna().sum()

customer_zip_code_prefix      0
latitude                    181
longitude                   181
dtype: int64

In [19]:
train = train.rename(
    columns={
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

validation = validation.rename(
    columns={
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

test = test.rename(
    columns={
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

### Seller Information

Seller information is obtained from the order items table because seller IDs are associated with individual order items rather than directly with the order-level dataset.

Since an order may contain items from multiple sellers, the seller information must be aggregated at the order level before being merged with the train, validation, and test datasets.


In [20]:
order_items = pd.read_csv("../Data/raw/olist_order_items_dataset.csv")

In [21]:
seller_per_order = (
    order_items
    .groupby("order_id")["seller_id"]
    .nunique()
)

seller_per_order.value_counts().sort_index()

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

### Seller Geographic Coordinates

Seller ZIP code prefixes are obtained from the order items and mapped to the seller dataset.

For orders containing multiple sellers, seller locations will be aggregated at the order level so that the final dataset remains one row per order.


In [22]:
order_seller = (
    order_items[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
    .merge(
        sellers[
            ["seller_id", "seller_zip_code_prefix"]
        ],
        on="seller_id",
        how="left"
    )
)

In [23]:
order_seller.head()

,order_id,seller_id,seller_zip_code_prefix
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900


In [24]:
order_seller = order_seller.merge(
    geo_by_zip,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

In [25]:
order_seller[
    ["seller_zip_code_prefix", "latitude", "longitude"]
].isna().sum()

seller_zip_code_prefix      0
latitude                  220
longitude                 220
dtype: int64

In [26]:
order_seller = order_seller.rename(
    columns={
        "latitude": "seller_latitude",
        "longitude": "seller_longitude"
    }
)

In [27]:
order_seller.columns.tolist()

['order_id',
 'seller_id',
 'seller_zip_code_prefix',
 'geolocation_zip_code_prefix',
 'seller_latitude',
 'seller_longitude']

In [28]:
train_seller = train[
    ["order_id", "customer_latitude", "customer_longitude"]
].merge(
    order_seller[
        ["order_id", "seller_latitude", "seller_longitude"]
    ],
    on="order_id",
    how="left"
)

In [29]:
train_seller.shape

(68347, 5)

### Customer-Seller Distance

The geographical distance between the customer and seller locations is calculated using the Haversine formula. This provides an approximate great-circle distance in kilometers based on the latitude and longitude coordinates.

The distance is calculated using only customer and seller location information available at prediction time.


In [30]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

In [31]:
train_seller["distance_km"] = haversine_distance(
    train_seller["customer_latitude"],
    train_seller["customer_longitude"],
    train_seller["seller_latitude"],
    train_seller["seller_longitude"]
)

In [32]:
train_seller["distance_km"].describe()

count    67997.000000
mean       614.267597
std        594.118591
min          0.000000
25%        219.962081
50%        448.427134
75%        810.937579
max       5338.619521
Name: distance_km, dtype: float64

### Aggregating Distance at Order Level

Some orders contain items from multiple sellers. To preserve the one-row-per-order structure required by the machine learning dataset, the customer-seller distances are aggregated by order.

The mean distance across the sellers associated with each order is used as the final `distance_km` feature.


In [33]:
train_distance = (
    train_seller
    .groupby("order_id", as_index=False)["distance_km"]
    .mean()
)

In [34]:
train_distance.shape

(67533, 2)

In [35]:
train = train.merge(
    train_distance,
    on="order_id",
    how="left"
)

In [36]:
train["distance_km"].isna().sum()

np.int64(344)

In [37]:
train["distance_km"].describe()

count    67189.000000
mean       614.743167
std        594.760614
min          0.000000
25%        219.962081
50%        448.619468
75%        810.729727
max       5338.619521
Name: distance_km, dtype: float64

In [38]:
validation_seller = validation[
    ["order_id", "customer_latitude", "customer_longitude"]
].merge(
    order_seller[
        ["order_id", "seller_latitude", "seller_longitude"]
    ],
    on="order_id",
    how="left"
)

validation_seller["distance_km"] = haversine_distance(
    validation_seller["customer_latitude"],
    validation_seller["customer_longitude"],
    validation_seller["seller_latitude"],
    validation_seller["seller_longitude"]
)

validation_distance = (
    validation_seller
    .groupby("order_id", as_index=False)["distance_km"]
    .mean()
)

validation = validation.merge(
    validation_distance,
    on="order_id",
    how="left"
)

In [39]:
validation["distance_km"].isna().sum()

np.int64(74)

In [40]:
test_seller = test[
    ["order_id", "customer_latitude", "customer_longitude"]
].merge(
    order_seller[
        ["order_id", "seller_latitude", "seller_longitude"]
    ],
    on="order_id",
    how="left"
)

test_seller["distance_km"] = haversine_distance(
    test_seller["customer_latitude"],
    test_seller["customer_longitude"],
    test_seller["seller_latitude"],
    test_seller["seller_longitude"]
)

test_distance = (
    test_seller
    .groupby("order_id", as_index=False)["distance_km"]
    .mean()
)

test = test.merge(
    test_distance,
    on="order_id",
    how="left"
)

In [41]:
test["distance_km"].isna().sum()

np.int64(58)

In [42]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 24)
Validation: (14471, 24)
Test: (14472, 24)


In [43]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'number_of_items',
 'total_price',
 'total_freight',
 'number_of_payments',
 'total_payment_value',
 'delivery_label',
 'purchase_hour',
 'purchase_weekday',
 'purchase_month',
 'estimated_delivery_days',
 'customer_state',
 'customer_zip_code_prefix',
 'geolocation_zip_code_prefix',
 'customer_latitude',
 'customer_longitude',
 'distance_km']

In [44]:
columns_to_drop = [
    "customer_zip_code_prefix",
    "geolocation_zip_code_prefix",
    "customer_latitude",
    "customer_longitude"
]

train = train.drop(columns=columns_to_drop)
validation = validation.drop(columns=columns_to_drop)
test = test.drop(columns=columns_to_drop)

In [45]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 20)
Validation: (14471, 20)
Test: (14472, 20)


## Feature Selection

Based on the EDA findings, the final feature set includes numerical, categorical, temporal, and geographical features.

The selected features are:

* **Order characteristics:** `number_of_items`, `total_price`, `total_freight`, `number_of_payments`, `total_payment_value`

* **Purchase timing:** `purchase_hour`, `purchase_weekday`, `purchase_month`

* **Estimated delivery:** `estimated_delivery_days`

* **Customer geography:** `customer_state`

* **Shipping distance:** `distance_km`

Identifiers, target variables, raw timestamps, and variables containing information that would only be available after the prediction point are excluded to prevent data leakage.

The selected categorical features will be encoded during preprocessing, while numerical features will be handled using the appropriate preprocessing transformations.

In [46]:
feature_cols = [
    "number_of_items",
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment_value",
    "purchase_hour",
    "purchase_weekday",
    "purchase_month",
    "estimated_delivery_days",
    "customer_state",
    "distance_km"
]

In [47]:
len(feature_cols), feature_cols

(11,
 ['number_of_items',
  'total_price',
  'total_freight',
  'number_of_payments',
  'total_payment_value',
  'purchase_hour',
  'purchase_weekday',
  'purchase_month',
  'estimated_delivery_days',
  'customer_state',
  'distance_km'])

### Separating Features and Target

The selected features are separated from the target variable before preprocessing.
`X` contains the input features used by the machine learning model, while `y` contains the target variable, `delivery_label`, which indicates whether an order was delivered late or on time.

In [48]:
X_train = train[feature_cols]
y_train = train["delivery_label"]

X_validation = validation[feature_cols]
y_validation = validation["delivery_label"]

X_test = test[feature_cols]
y_test = test["delivery_label"]

In [49]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (67533, 11)
y_train: (67533,)
X_validation: (14471, 11)
y_validation: (14471,)
X_test: (14472, 11)
y_test: (14472,)


### Inspecting Feature Types and Missing Values

Before applying preprocessing, the selected features are inspected to identify their data types and missing values.

This allows the appropriate transformation to be applied to each feature group.

In [50]:
X_train.dtypes

number_of_items            float64
total_price                float64
total_freight              float64
number_of_payments         float64
total_payment_value        float64
purchase_hour                int32
purchase_weekday               str
purchase_month                 str
estimated_delivery_days    float64
customer_state                 str
distance_km                float64
dtype: object

In [51]:
X_train.isna().sum()

number_of_items              0
total_price                  0
total_freight                0
number_of_payments           1
total_payment_value          1
purchase_hour                0
purchase_weekday             0
purchase_month               0
estimated_delivery_days      0
customer_state               0
distance_km                344
dtype: int64

### Defining Feature Groups

The selected features are divided into numerical and categorical groups so that each group can receive the appropriate preprocessing.

Numerical features will be imputed using statistics learned from the training data. Categorical features will be encoded using an encoder fitted only on the training split.

In [52]:
numeric_features = [
    "number_of_items",
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment_value",
    "purchase_hour",
    "estimated_delivery_days",
    "distance_km"
]

categorical_features = [
    "purchase_weekday",
    "purchase_month",
    "customer_state"
]

In [53]:
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 8
Categorical features: 3


### Handling Missing Numerical Values

Missing values in numerical features are handled using median imputation.

The imputer is fitted only on the training data to prevent data leakage. The learned median values are then applied to the validation and test sets.

In [54]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="median")

In [55]:
numeric_features = [
    "number_of_items",
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment_value",
    "purchase_hour",
    "estimated_delivery_days",
    "distance_km"
]

numeric_imputer.fit(X_train[numeric_features])

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['number_of_items','total_price','total_freight',...,'purchase_hour', 'estimated_delivery_days','distance_km']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](8,)","[ 1. , 85. , 16.79,..., 15. , 23.62,448.62]"


In [56]:
X_train[numeric_features] = numeric_imputer.transform(
    X_train[numeric_features]
)

In [57]:
X_validation[numeric_features] = numeric_imputer.transform(
    X_validation[numeric_features]
)

X_test[numeric_features] = numeric_imputer.transform(
    X_test[numeric_features]
)

In [58]:
X_train[numeric_features].isna().sum()

number_of_items            0
total_price                0
total_freight              0
number_of_payments         0
total_payment_value        0
purchase_hour              0
estimated_delivery_days    0
distance_km                0
dtype: int64

In [59]:
X_validation[numeric_features].isna().sum()

number_of_items            0
total_price                0
total_freight              0
number_of_payments         0
total_payment_value        0
purchase_hour              0
estimated_delivery_days    0
distance_km                0
dtype: int64

In [60]:
X_test[numeric_features].isna().sum()

number_of_items            0
total_price                0
total_freight              0
number_of_payments         0
total_payment_value        0
purchase_hour              0
estimated_delivery_days    0
distance_km                0
dtype: int64

In [61]:
from sklearn.preprocessing import OneHotEncoder

In [62]:
categorical_features = [
    "purchase_weekday",
    "purchase_month",
    "customer_state"
]

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [63]:
categorical_encoder.fit(X_train[categorical_features])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [64]:
X_train_encoded = categorical_encoder.transform(
    X_train[categorical_features]
)

In [65]:
X_train_encoded.shape

(67533, 53)

In [66]:
X_validation_encoded = categorical_encoder.transform(
    X_validation[categorical_features]
)

X_test_encoded = categorical_encoder.transform(
    X_test[categorical_features]
)

In [67]:
print("Train:", X_train_encoded.shape)
print("Validation:", X_validation_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (67533, 53)
Validation: (14471, 53)
Test: (14472, 53)


In [68]:
encoded_feature_names = categorical_encoder.get_feature_names_out(
    categorical_features
)

encoded_feature_names

array(['purchase_weekday_Friday', 'purchase_weekday_Monday',
       'purchase_weekday_Saturday', 'purchase_weekday_Sunday',
       'purchase_weekday_Thursday', 'purchase_weekday_Tuesday',
       'purchase_weekday_Wednesday', 'purchase_month_2016-09',
       'purchase_month_2016-10', 'purchase_month_2016-12',
       'purchase_month_2017-01', 'purchase_month_2017-02',
       'purchase_month_2017-03', 'purchase_month_2017-04',
       'purchase_month_2017-05', 'purchase_month_2017-06',
       'purchase_month_2017-07', 'purchase_month_2017-08',
       'purchase_month_2017-09', 'purchase_month_2017-10',
       'purchase_month_2017-11', 'purchase_month_2017-12',
       'purchase_month_2018-01', 'purchase_month_2018-02',
       'purchase_month_2018-03', 'purchase_month_2018-04',
       'customer_state_AC', 'customer_state_AL', 'customer_state_AM',
       'customer_state_AP', 'customer_state_BA', 'customer_state_CE',
       'customer_state_DF', 'customer_state_ES', 'customer_state_GO',
       '

In [69]:
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)
X_train_encoded_df.shape

(67533, 53)

In [70]:
X_validation_encoded_df = pd.DataFrame(
    X_validation_encoded,
    columns=encoded_feature_names,
    index=X_validation.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

In [71]:
print("Train:", X_train_encoded_df.shape)
print("Validation:", X_validation_encoded_df.shape)
print("Test:", X_test_encoded_df.shape)

Train: (67533, 53)
Validation: (14471, 53)
Test: (14472, 53)


In [72]:
X_train_final = pd.concat(
    [
        X_train[numeric_features],
        X_train_encoded_df
    ],
    axis=1
)
X_train_final.shape

(67533, 61)

In [73]:
X_validation_final = pd.concat(
    [
        X_validation[numeric_features],
        X_validation_encoded_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test[numeric_features],
        X_test_encoded_df
    ],
    axis=1
)

In [74]:
print("X_train:", X_train_final.shape)
print("X_validation:", X_validation_final.shape)
print("X_test:", X_test_final.shape)

X_train: (67533, 61)
X_validation: (14471, 61)
X_test: (14472, 61)


In [75]:
print("Train missing:", X_train_final.isna().sum().sum())
print("Validation missing:", X_validation_final.isna().sum().sum())
print("Test missing:", X_test_final.isna().sum().sum())

Train missing: 0
Validation missing: 0
Test missing: 0


In [76]:
feature_list = X_train_final.columns.tolist()

print("Number of features:", len(feature_list))
print(feature_list)

Number of features: 61
['number_of_items', 'total_price', 'total_freight', 'number_of_payments', 'total_payment_value', 'purchase_hour', 'estimated_delivery_days', 'distance_km', 'purchase_weekday_Friday', 'purchase_weekday_Monday', 'purchase_weekday_Saturday', 'purchase_weekday_Sunday', 'purchase_weekday_Thursday', 'purchase_weekday_Tuesday', 'purchase_weekday_Wednesday', 'purchase_month_2016-09', 'purchase_month_2016-10', 'purchase_month_2016-12', 'purchase_month_2017-01', 'purchase_month_2017-02', 'purchase_month_2017-03', 'purchase_month_2017-04', 'purchase_month_2017-05', 'purchase_month_2017-06', 'purchase_month_2017-07', 'purchase_month_2017-08', 'purchase_month_2017-09', 'purchase_month_2017-10', 'purchase_month_2017-11', 'purchase_month_2017-12', 'purchase_month_2018-01', 'purchase_month_2018-02', 'purchase_month_2018-03', 'purchase_month_2018-04', 'customer_state_AC', 'customer_state_AL', 'customer_state_AM', 'customer_state_AP', 'customer_state_BA', 'customer_state_CE', 'cus

In [77]:
# import json
# import os

# feature_list_path = "../Artifacts/feature_list.json"

# with open(feature_list_path, "w") as f:
#     json.dump(feature_list, f, indent=2)

# print(f"Feature list saved to: {feature_list_path}")

In [78]:

# train_features = X_train_final.copy()
# validation_features = X_validation_final.copy()
# test_features = X_test_final.copy()

# train_features["split"] = "train"
# validation_features["split"] = "validation"
# test_features["split"] = "test"

# final_feature_table = pd.concat(
#     [train_features, validation_features, test_features],
#     axis=0,
#     ignore_index=True
# )

# final_feature_table_path = "../Artifacts/final_feature_table.csv"

# final_feature_table.to_csv(
#     final_feature_table_path,
#     index=False
# )

# print("Final feature table shape:", final_feature_table.shape)
# print(f"Final feature table saved to: {final_feature_table_path}")

In [79]:
# print(final_feature_table["split"].value_counts())
# print()
# print("Feature columns:", len(feature_list))
# print("Final table shape:", final_feature_table.shape)

In [80]:
# print(
#     X_train_final.columns.equals(X_validation_final.columns)
# )

# print(
#     X_train_final.columns.equals(X_test_final.columns)
# )

In [81]:
# import os
# print(
#     "Feature list exists:",
#     os.path.exists("../Artifacts/feature_list.json")
# )

# print(
#     "Final feature table exists:",
#     os.path.exists("../Artifacts/final_feature_table.csv")
# )

# print("\nRequired preprocessing artifacts:")

# artifact_paths = {
#     "Numeric imputer": "../Artifacts/models/numeric_imputer.joblib",
#     "Categorical encoder": "../Artifacts/models/categorical_encoder.joblib",
#     "Scaler": "../Artifacts/models/scaler.joblib"
# }

# for name, path in artifact_paths.items():
#     print(f"{name} exists:", os.path.exists(path))

In [82]:
# print("Feature list length:", len(feature_list))
# print("Feature table shape:", final_feature_table.shape)

# print("\nFirst 5 features:")
# print(feature_list[:5])

# print("\nSplits:")
# print(final_feature_table["split"].value_counts())

## Data Splitting and Preprocessing

The dataset was divided into three subsets: training, validation, and test sets.

* **Training set:** 67,533 rows
* **Validation set:** 14,471 rows
* **Test set:** 14,472 rows

Feature engineering was performed to create predictive features related to order characteristics, purchase timing, estimated delivery duration, customer location, and geographic distance.

The final feature set consists of **61 features**:

* **8 numerical features**
* **53 one-hot encoded categorical features**

Missing values in numerical features were handled using median imputation, with the imputation statistics learned from the training set only. Categorical variables were transformed using One-Hot Encoding, with unknown categories handled safely in the validation and test sets.

The same fitted preprocessing transformations were applied consistently to the validation and test sets without refitting.

The resulting feature datasets are ready to be used in Notebook 6 for model training, tuning, and evaluation.

In [83]:
y_train.value_counts()

delivery_label
on_time    62242
late        5291
Name: count, dtype: int64

In [84]:
y_train.value_counts(normalize=True) * 100

delivery_label
on_time    92.165312
late        7.834688
Name: proportion, dtype: float64

In [85]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_numeric_scaled = scaler.fit_transform(
    X_train[numeric_features]
)

X_validation_numeric_scaled = scaler.transform(
    X_validation[numeric_features]
)

X_test_numeric_scaled = scaler.transform(
    X_test[numeric_features]
)

In [86]:
print(X_train_numeric_scaled.shape)
print(X_validation_numeric_scaled.shape)
print(X_test_numeric_scaled.shape)

(67533, 8)
(14471, 8)
(14472, 8)


In [87]:
X_train_numeric_scaled_df = pd.DataFrame(
    X_train_numeric_scaled,
    columns=numeric_features,
    index=X_train.index
)

X_validation_numeric_scaled_df = pd.DataFrame(
    X_validation_numeric_scaled,
    columns=numeric_features,
    index=X_validation.index
)

X_test_numeric_scaled_df = pd.DataFrame(
    X_test_numeric_scaled,
    columns=numeric_features,
    index=X_test.index
)

In [88]:
X_train_final = pd.concat(
    [X_train_numeric_scaled_df, X_train_encoded_df],
    axis=1
)

X_validation_final = pd.concat(
    [X_validation_numeric_scaled_df, X_validation_encoded_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test_numeric_scaled_df, X_test_encoded_df],
    axis=1
)

In [89]:
print("Train:", X_train_final.shape)
print("Validation:", X_validation_final.shape)
print("Test:", X_test_final.shape)

Train: (67533, 61)
Validation: (14471, 61)
Test: (14472, 61)


In [90]:
import json
import os

feature_list_path = "../Artifacts/feature_list.json"

with open(feature_list_path, "w") as f:
    json.dump(feature_list, f, indent=2)

print(f"Feature list saved to: {feature_list_path}")

Feature list saved to: ../Artifacts/feature_list.json


In [91]:
train_features = X_train_final.copy()
validation_features = X_validation_final.copy()
test_features = X_test_final.copy()

train_features["split"] = "train"
validation_features["split"] = "validation"
test_features["split"] = "test"

final_feature_table = pd.concat(
    [train_features, validation_features, test_features],
    axis=0,
    ignore_index=True
)

final_feature_table_path = "../Artifacts/final_feature_table.csv"

final_feature_table.to_csv(
    final_feature_table_path,
    index=False
)

print("Final feature table shape:", final_feature_table.shape)
print(f"Final feature table saved to: {final_feature_table_path}")

Final feature table shape: (96476, 62)
Final feature table saved to: ../Artifacts/final_feature_table.csv


In [92]:
print(final_feature_table["split"].value_counts())
print()
print("Feature columns:", len(feature_list))
print("Final table shape:", final_feature_table.shape)

split
train         67533
test          14472
validation    14471
Name: count, dtype: int64

Feature columns: 61
Final table shape: (96476, 62)


In [93]:
print(
    X_train_final.columns.equals(X_validation_final.columns)
)

print(
    X_train_final.columns.equals(X_test_final.columns)
)

True
True


In [94]:
print(
    "Feature list exists:",
    os.path.exists("../Artifacts/feature_list.json")
)

print(
    "Final feature table exists:",
    os.path.exists("../Artifacts/final_feature_table.csv")
)

print("\nRequired preprocessing artifacts:")

artifact_paths = {
    "Numeric imputer": "../Artifacts/models/numeric_imputer.joblib",
    "Categorical encoder": "../Artifacts/models/categorical_encoder.joblib",
    "Scaler": "../Artifacts/models/scaler.joblib"
}

for name, path in artifact_paths.items():
    print(f"{name} exists:", os.path.exists(path))

Feature list exists: True
Final feature table exists: True

Required preprocessing artifacts:
Numeric imputer exists: True
Categorical encoder exists: True
Scaler exists: True


In [95]:
print("Feature list length:", len(feature_list))
print("Feature table shape:", final_feature_table.shape)

print("\nFirst 5 features:")
print(feature_list[:5])

print("\nSplits:")
print(final_feature_table["split"].value_counts())

Feature list length: 61
Feature table shape: (96476, 62)

First 5 features:
['number_of_items', 'total_price', 'total_freight', 'number_of_payments', 'total_payment_value']

Splits:
split
train         67533
test          14472
validation    14471
Name: count, dtype: int64


In [97]:
check = pd.read_csv(
    "../Artifacts/final_feature_table.csv"
)

print(check.shape)
print(check.head())

(96476, 62)
   number_of_items  total_price  total_freight  number_of_payments  \
0         3.430235    -0.004457      -0.687732            -0.11989   
1        -0.262316    -0.516529      -0.334222            -0.11989   
2        -0.262316    -0.555518      -0.252720            -0.11989   
3        -0.262316    -0.557467      -0.406724            -0.11989   
4        -0.262316    -0.484411      -0.250220            -0.11989   

   total_payment_value  purchase_hour  estimated_delivery_days  distance_km  \
0            -0.251905      -0.519162                -0.761504    -0.080790   
1            -0.525992      -1.079511                -0.125083     0.159865   
2            -0.555721       0.227972                 1.208737     0.508994   
3            -0.571961       1.161888                 3.431350    -0.431987   
4            -0.487397       1.161888                 3.928943     0.344227   

   purchase_weekday_Friday  purchase_weekday_Monday  ...  customer_state_RJ  \
0            

## Conclusion

In this notebook, the Olist dataset was prepared for machine learning through feature engineering and preprocessing.

The selected features were created using information available at prediction time, while variables containing future delivery information were excluded to prevent data leakage.

The data was divided into training, validation, and test sets. Numerical features were processed using transformations fitted on the training data only, while categorical features were encoded consistently across all splits.

The final preprocessed feature table contains 61 features across 96,476 observations, with the original train, validation, and test splits preserved.

The fitted preprocessing transformers, final feature table, and feature list were saved as reusable artifacts for subsequent model training and evaluation in Notebook 6.

Model training, tuning, and final evaluation are performed separately in Notebook 6.

In [96]:
# Final N5 verification

print("Final feature table:", final_feature_table.shape)
print("Number of features:", len(feature_list))

print("\nRequired artifacts:")
print("Feature list:", os.path.exists("../Artifacts/feature_list.json"))
print("Final feature table:", os.path.exists("../Artifacts/final_feature_table.csv"))
print("Numeric imputer:", os.path.exists("../Artifacts/models/numeric_imputer.joblib"))
print("Categorical encoder:", os.path.exists("../Artifacts/models/categorical_encoder.joblib"))
print("Scaler:", os.path.exists("../Artifacts/models/scaler.joblib"))

Final feature table: (96476, 62)
Number of features: 61

Required artifacts:
Feature list: True
Final feature table: True
Numeric imputer: True
Categorical encoder: True
Scaler: True
